# 🔄 Recursion & Backtracking — Trust the Process

Recursion and backtracking handle problems where you need to explore **all possibilities**:
all subsets, all permutations, all combinations, all valid placements.

**Recognition Triggers:**
- "all subsets / combinations / permutations" → Backtracking
- "generate all valid X" → Backtracking
- "find if a path/solution exists" → DFS/Backtracking
- "N-Queens / Sudoku" → Backtracking with constraint checking

---
## 🧠 Recursion Thinking Model: Trust the Recursion

### The Recipe
1. **Define the function signature** — what does it take and return?
2. **Write the base case** — the simplest input that has a known answer
3. **Write the recursive case** — assume the function works for smaller inputs ("trust the recursion"), use that to build the current answer

### The Golden Rule
> **Do NOT trace through the recursion mentally.** Trust that it works for smaller inputs and focus on ONE level.

In [ ]:
# Classic recursion examples

def factorial(n):
    """Base: 0! = 1. Recursive: n! = n * (n-1)!"""
    if n <= 1: return 1
    return n * factorial(n - 1)

def fibonacci(n):
    """Base: fib(0)=0, fib(1)=1. Recursive: fib(n) = fib(n-1) + fib(n-2)
    WARNING: O(2^n) without memoization! Always use @lru_cache in contests."""
    if n <= 1: return n
    return fibonacci(n - 1) + fibonacci(n - 2)

def power(base, exp):
    """O(log n) fast exponentiation."""
    if exp == 0: return 1
    half = power(base, exp // 2)
    if exp % 2 == 0:
        return half * half
    return half * half * base

print(f"5! = {factorial(5)}")          # 120
print(f"fib(10) = {fibonacci(10)}")    # 55
print(f"2^10 = {power(2, 10)}")        # 1024

---
## 📋 The Backtracking Template

```python
def backtrack(candidates, current, start, result):
    # 1. BASE CASE: found a valid solution
    if is_solution(current):
        result.append(current[:])
        return
    
    # 2. EXPLORE: try each candidate
    for i in range(start, len(candidates)):
        # 3. PRUNE: skip invalid choices early
        if not is_valid(candidates[i]):
            continue
        
        # 4. CHOOSE: add to current solution
        current.append(candidates[i])
        
        # 5. RECURSE: explore with this choice
        backtrack(candidates, current, i + 1, result)
        
        # 6. UNCHOOSE: backtrack (remove the choice)
        current.pop()
```

---
## 🔥 Problem: Subsets

In [ ]:
def subsets(nums):
    """Generate all 2^n subsets. Time: O(n * 2^n), Space: O(n * 2^n)"""
    result = []
    
    def backtrack(start, current):
        result.append(current[:])
        for i in range(start, len(nums)):
            current.append(nums[i])
            backtrack(i + 1, current)
            current.pop()
    
    backtrack(0, [])
    return result

# Iterative version (for comparison)
def subsets_iterative(nums):
    result = [[]]
    for num in nums:
        result += [subset + [num] for subset in result]
    return result

# Bitmask version
def subsets_bitmask(nums):
    n = len(nums)
    return [[nums[j] for j in range(n) if mask & (1 << j)] for mask in range(1 << n)]

print("Subsets of [1,2,3]:")
for s in subsets([1, 2, 3]):
    print(f"  {s}")

---
## 🔥 Problem: Combinations

In [ ]:
def combine(n, k):
    """All combinations of k numbers from [1, n]. Time: O(C(n,k) * k)"""
    result = []
    
    def backtrack(start, current):
        if len(current) == k:
            result.append(current[:])
            return
        # Pruning: need (k - len(current)) more elements
        for i in range(start, n + 1 - (k - len(current)) + 1):
            current.append(i)
            backtrack(i + 1, current)
            current.pop()
    
    backtrack(1, [])
    return result

print("C(4, 2) =", combine(4, 2))
# [[1,2], [1,3], [1,4], [2,3], [2,4], [3,4]]

---
## 🔥 Problem: Permutations

In [ ]:
def permutations(nums):
    """All n! permutations. Time: O(n * n!)"""
    result = []
    
    def backtrack(current, remaining):
        if not remaining:
            result.append(current[:])
            return
        for i in range(len(remaining)):
            current.append(remaining[i])
            backtrack(current, remaining[:i] + remaining[i+1:])
            current.pop()
    
    backtrack([], nums)
    return result

# Using swap (in-place, more efficient)
def permutations_swap(nums):
    result = []
    def backtrack(start):
        if start == len(nums):
            result.append(nums[:])
            return
        for i in range(start, len(nums)):
            nums[start], nums[i] = nums[i], nums[start]
            backtrack(start + 1)
            nums[start], nums[i] = nums[i], nums[start]
    backtrack(0)
    return result

print("Permutations of [1,2,3]:")
for p in permutations([1, 2, 3]):
    print(f"  {p}")

---
## 🔥 Problem: Combination Sum

In [ ]:
def combination_sum(candidates, target):
    """Find all unique combinations that sum to target.
    Same number CAN be used multiple times.
    Time: depends on branching, roughly O(n^(target/min(candidates)))"""
    result = []
    candidates.sort()
    
    def backtrack(start, current, remaining):
        if remaining == 0:
            result.append(current[:])
            return
        for i in range(start, len(candidates)):
            if candidates[i] > remaining:  # Pruning!
                break
            current.append(candidates[i])
            backtrack(i, current, remaining - candidates[i])  # i, not i+1 (reuse!)
            current.pop()
    
    backtrack(0, [], target)
    return result

print("Combination Sum [2,3,6,7], target=7:")
for combo in combination_sum([2, 3, 6, 7], 7):
    print(f"  {combo}")
# [[2,2,3], [7]]

---
## 🔥 Problem: Word Search

In [ ]:
def word_search(board, word):
    """DFS + backtracking on a grid. Time: O(m*n*4^L) where L = len(word)"""
    rows, cols = len(board), len(board[0])
    
    def dfs(r, c, i):
        if i == len(word):
            return True
        if r < 0 or r >= rows or c < 0 or c >= cols:
            return False
        if board[r][c] != word[i]:
            return False
        
        # Mark as visited
        temp = board[r][c]
        board[r][c] = '#'
        
        # Explore 4 directions
        found = (dfs(r+1, c, i+1) or dfs(r-1, c, i+1) or
                 dfs(r, c+1, i+1) or dfs(r, c-1, i+1))
        
        # Backtrack
        board[r][c] = temp
        return found
    
    for r in range(rows):
        for c in range(cols):
            if dfs(r, c, 0):
                return True
    return False

board = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]]
print("Word Search:")
print(f"  'ABCCED': {word_search([row[:] for row in board], 'ABCCED')}")  # True
print(f"  'SEE': {word_search([row[:] for row in board], 'SEE')}")        # True
print(f"  'ABCB': {word_search([row[:] for row in board], 'ABCB')}")     # False

---
## 🔥 Problem: N-Queens

In [ ]:
def solve_n_queens(n):
    """Place n queens on n×n board. No two queens attack each other.
    Time: O(n!), Space: O(n²)"""
    result = []
    cols = set()
    pos_diag = set()  # (r + c) is constant on positive diagonals
    neg_diag = set()  # (r - c) is constant on negative diagonals
    board = [['.' for _ in range(n)] for _ in range(n)]
    
    def backtrack(row):
        if row == n:
            result.append([''.join(r) for r in board])
            return
        
        for col in range(n):
            if col in cols or (row + col) in pos_diag or (row - col) in neg_diag:
                continue  # Pruning: this position is under attack
            
            # Place queen
            cols.add(col)
            pos_diag.add(row + col)
            neg_diag.add(row - col)
            board[row][col] = 'Q'
            
            backtrack(row + 1)
            
            # Remove queen (backtrack)
            cols.remove(col)
            pos_diag.remove(row + col)
            neg_diag.remove(row - col)
            board[row][col] = '.'
    
    backtrack(0)
    return result

print(f"N-Queens (n=4): {len(solve_n_queens(4))} solutions")
for solution in solve_n_queens(4):
    for row in solution:
        print(f"  {row}")
    print()

---
## 🔥 Problem: Sudoku Solver

In [ ]:
def solve_sudoku(board):
    """Fill the board in-place. Backtracking with constraint propagation."""
    from collections import defaultdict
    
    rows = defaultdict(set)
    cols = defaultdict(set)
    boxes = defaultdict(set)
    empty = []
    
    # Initialize constraints
    for r in range(9):
        for c in range(9):
            if board[r][c] == '.':
                empty.append((r, c))
            else:
                val = board[r][c]
                rows[r].add(val)
                cols[c].add(val)
                boxes[(r//3, c//3)].add(val)
    
    def backtrack(idx):
        if idx == len(empty):
            return True  # Solved!
        
        r, c = empty[idx]
        box_key = (r // 3, c // 3)
        
        for digit in '123456789':
            if digit in rows[r] or digit in cols[c] or digit in boxes[box_key]:
                continue
            
            # Place digit
            board[r][c] = digit
            rows[r].add(digit)
            cols[c].add(digit)
            boxes[box_key].add(digit)
            
            if backtrack(idx + 1):
                return True
            
            # Backtrack
            board[r][c] = '.'
            rows[r].remove(digit)
            cols[c].remove(digit)
            boxes[box_key].remove(digit)
        
        return False
    
    backtrack(0)
    return board

print("Sudoku Solver: (solving a sample board)")
board = [
    ["5","3",".",".","7",".",".",".","."],
    ["6",".",".","1","9","5",".",".","."],
    [".","9","8",".",".",".",".","6","."],
    ["8",".",".",".","6",".",".",".","3"],
    ["4",".",".","8",".","3",".",".","1"],
    ["7",".",".",".","2",".",".",".","6"],
    [".","6",".",".",".",".","2","8","."],
    [".",".",".","4","1","9",".",".","5"],
    [".",".",".",".","8",".",".","7","9"]
]
solve_sudoku(board)
for row in board:
    print("  ", " ".join(row))

---
## ⚡ Pruning Strategies & Python Tips

### Pruning (making backtracking faster)
1. **Sort candidates** — try smaller values first, fail faster
2. **Early termination** — if remaining sum < 0, stop
3. **Skip duplicates** — `if i > start and candidates[i] == candidates[i-1]: continue`
4. **Constraint sets** — like N-Queens (cols, diags), check O(1) instead of scanning

### Python-Specific Tips
```python
import sys
sys.setrecursionlimit(10**6)  # Default is 1000 — TOO LOW for contests!
```

### Iterative DFS as Alternative
When recursion depth might exceed limits, convert to iterative with explicit stack.

In [ ]:
# Iterative DFS for subsets (no recursion limit issues)
def subsets_iterative_stack(nums):
    result = []
    stack = [(0, [])]  # (start_index, current_subset)
    
    while stack:
        start, current = stack.pop()
        result.append(current)
        for i in range(start, len(nums)):
            stack.append((i + 1, current + [nums[i]]))
    
    return result

print("Iterative subsets:", subsets_iterative_stack([1, 2, 3]))

---
## 🏆 Summary

| Problem | Time | Key Technique |
|---------|------|---------------|
| Subsets | O(n × 2ⁿ) | Include/exclude each element |
| Combinations | O(C(n,k) × k) | Start index + length check |
| Permutations | O(n × n!) | Swap or remaining list |
| Combination Sum | O(varies) | Allow reuse (pass i, not i+1) |
| Word Search | O(mn × 4ᴸ) | Grid DFS + visited marking |
| N-Queens | O(n!) | Row-by-row, constraint sets |
| Sudoku Solver | O(9^(empty)) | Cell-by-cell, constraint sets |

**Key Pattern:** Choose → Recurse → Unchoose (backtrack). Always.